In [3]:
import pandas as pd

In [20]:
# read the csv
df = pd.read_csv('signup.csv')
df

,date,name,status,gender,type
0,2026-06-06,Andy Connor,confirmed,M,player
1,2026-06-06,Ronald Boyce,confirmed,M,player
2,2026-06-06,George Grimshaw,confirmed,M,player
3,2026-06-06,Stacy Jarvis,confirmed,F,player
4,2026-06-06,Leigh Gale,confirmed,F,player
5,2026-06-06,Kathryn Hartley,confirmed,F,player
6,2026-06-06,Oliver Castle,confirmed,NB,player
7,2026-06-06,Nisha Fellows,confirmed,F,player
8,2026-06-06,Molly Farrow,confirmed,F,player
9,2026-06-06,Andy Joseph,confirmed,M,player


In [21]:
df.columns = df.columns.str.strip()

# separate waitlist and cancelled 
confirmed_players = df[df["status"].str.strip() == "confirmed"].copy()
waitlist_players = df[df["status"].str.strip() == "waitlist"].copy()
cancelled_players = df[df["status"].str.strip() == "cancelled"].copy()

confirmed_players.sample()

,date,name,status,gender,type
16,2026-06-06,Beatrice Nelson,confirmed,NB,player


In [22]:
# add the extra columns ready
confirmed_players['games_played'] = 0
confirmed_players['rounds_rested'] = 0
confirmed_players['current_state'] = "waiting"

confirmed_players.sample()

,date,name,status,gender,type,games_played,rounds_rested,current_state
19,2026-06-06,Joe Hurley,confirmed,M,coach,0,0,waiting


In [23]:
# for now lets assume that 16 players have self selected and
# assigned themselves courts
# split into resting/playing?
playing_players = confirmed_players.sample(n=16)

resting_players = confirmed_players[
    ~confirmed_players.index.isin(playing_players.index)
]

In [24]:
playing_players

,date,name,status,gender,type,games_played,rounds_rested,current_state
5,2026-06-06,Kathryn Hartley,confirmed,F,player,0,0,waiting
2,2026-06-06,George Grimshaw,confirmed,M,player,0,0,waiting
1,2026-06-06,Ronald Boyce,confirmed,M,player,0,0,waiting
14,2026-06-06,Phillip Pike,confirmed,M,player,0,0,waiting
10,2026-06-06,Andrzej Aspinall,confirmed,M,player,0,0,waiting
16,2026-06-06,Beatrice Nelson,confirmed,NB,player,0,0,waiting
15,2026-06-06,Nicholas McGowan,confirmed,M,player,0,0,waiting
11,2026-06-06,Saima Hague,confirmed,F,player,0,0,waiting
7,2026-06-06,Nisha Fellows,confirmed,F,player,0,0,waiting
8,2026-06-06,Molly Farrow,confirmed,F,player,0,0,waiting


In [25]:
# temp player UUID
confirmed_players["player_id"] = range(len(confirmed_players))

In [26]:
# generate round 1

# shuffle
playing_players = playing_players.sample(frac=1, random_state=42).reset_index(drop=True)

n_courts = 4
players_per_court = 4

# court list
courts = []

# add players to court 
for court_number in range(n_courts):
    start = court_number * players_per_court
    court_players = playing_players.iloc[start:start + players_per_court]

    courts.append({
        "court": court_number + 1,
        "players": court_players["name"].tolist()
    })

for court in courts:
    print(f"\nCourt {court['court']}")
    print(court["players"])


Court 1
['Kathryn Hartley', 'George Grimshaw', 'Beatrice Nelson', 'Laurence Lomax']

Court 2
['Tracy Benson', 'Sue Elliott', 'Nisha Fellows', 'Molly Farrow']

Court 3
['Ronald Boyce', 'Ronnie Jones', 'Andrzej Aspinall', 'Saima Hague']

Court 4
['Leigh Gale', 'Andy Connor', 'Phillip Pike', 'Nicholas McGowan']


In [27]:
# split each team into two sides
matches = []

for court in courts:
    players = court["players"]

    match = {
        "court": court["court"],
        "team1": players[:2],
        "team2": players[2:]
    }

    matches.append(match)

for match in matches:
    print(f"\nCourt {match['court']}")
    print(f"{match['team1'][0]} + {match['team1'][1]}")
    print("vs")
    print(f"{match['team2'][0]} + {match['team2'][1]}")


Court 1
Kathryn Hartley + George Grimshaw
vs
Beatrice Nelson + Laurence Lomax

Court 2
Tracy Benson + Sue Elliott
vs
Nisha Fellows + Molly Farrow

Court 3
Ronald Boyce + Ronnie Jones
vs
Andrzej Aspinall + Saima Hague

Court 4
Leigh Gale + Andy Connor
vs
Phillip Pike + Nicholas McGowan


In [28]:
# update played/rested and status
# Update players who played

confirmed_players["current_status"] = "waiting"

confirmed_players.loc[
    confirmed_players["name"].isin(playing_players["name"]),
    "games_played"
] += 1

# Update players who rested
confirmed_players.loc[
    confirmed_players["name"].isin(resting_players["name"]),
    "rounds_rested"
] += 1

# Mark active players
confirmed_players.loc[
    confirmed_players["name"].isin(playing_players["name"]),
    "current_status"
] = "playing"

In [29]:
# check 
confirmed_players

,date,name,status,gender,type,games_played,rounds_rested,current_state,player_id,current_status
0,2026-06-06,Andy Connor,confirmed,M,player,1,0,waiting,0,playing
1,2026-06-06,Ronald Boyce,confirmed,M,player,1,0,waiting,1,playing
2,2026-06-06,George Grimshaw,confirmed,M,player,1,0,waiting,2,playing
3,2026-06-06,Stacy Jarvis,confirmed,F,player,0,1,waiting,3,waiting
4,2026-06-06,Leigh Gale,confirmed,F,player,1,0,waiting,4,playing
5,2026-06-06,Kathryn Hartley,confirmed,F,player,1,0,waiting,5,playing
6,2026-06-06,Oliver Castle,confirmed,NB,player,0,1,waiting,6,waiting
7,2026-06-06,Nisha Fellows,confirmed,F,player,1,0,waiting,7,playing
8,2026-06-06,Molly Farrow,confirmed,F,player,1,0,waiting,8,playing
9,2026-06-06,Andy Joseph,confirmed,M,player,0,1,waiting,9,waiting


In [30]:
def choose_players_for_round(players_df, courts=4):
    players_needed = courts * 4

    sorted_players = players_df.sort_values(
        by=["games_played", "rounds_rested"],
        ascending=[True, False]
    )

    playing_players = sorted_players.head(players_needed).copy()

    resting_players = players_df[
        ~players_df["name"].isin(playing_players["name"])
    ].copy()

    return playing_players, resting_players

In [31]:
playing_players, resting_players = choose_players_for_round(
    confirmed_players,
    courts=4
)

In [32]:
confirmed_players["current_status"] = "resting"

confirmed_players.loc[
    confirmed_players["name"].isin(playing_players["name"]),
    "current_status"
] = "playing"

confirmed_players.loc[
    confirmed_players["name"].isin(playing_players["name"]),
    "games_played"
] += 1

confirmed_players.loc[
    confirmed_players["name"].isin(resting_players["name"]),
    "rounds_rested"
] += 1

In [34]:
confirmed_players[["name", "gender", "games_played", "rounds_rested", "current_status"]].sort_values(
    ["games_played", "rounds_rested"],
    ascending=[True, False]
)

,name,gender,games_played,rounds_rested,current_status
3,Stacy Jarvis,F,1,1,playing
6,Oliver Castle,NB,1,1,playing
9,Andy Joseph,M,1,1,playing
13,Tracy Benson,F,1,1,resting
14,Phillip Pike,M,1,1,resting
15,Nicholas McGowan,M,1,1,resting
16,Beatrice Nelson,NB,1,1,resting
17,Jeff Chamberlain,M,1,1,playing
18,Laurence Lomax,M,1,1,resting
19,Joe Hurley,M,1,1,playing


In [35]:
# now lets take into account that courts finish at diff times
def choose_players_for_court(players_df, n_players=4):
    available_players = players_df[
        players_df["current_status"] == "waiting"
    ].copy()

    selected_players = available_players.sort_values(
        by=["games_played", "rounds_rested"],
        ascending=[True, False]
    ).head(n_players)

    return selected_players

In [36]:
selected_players = choose_players_for_court(confirmed_players)

confirmed_players.loc[
    confirmed_players["name"].isin(selected_players["name"]),
    "current_status"
] = "playing"

In [37]:
confirmed_players.loc[
    confirmed_players["name"].isin(selected_players["name"]),
    "current_status"
] = "waiting"

confirmed_players.loc[
    confirmed_players["name"].isin(selected_players["name"]),
    "games_played"
] += 1

In [38]:
confirmed_players.loc[
    confirmed_players["current_status"] == "waiting",
    "rounds_rested"
] += 1